In [8]:
import pandas as pd
import numpy as np

def identify_isotopes(csv_file, min_animals=12, min_score=60):
    """
    Identify m/z values as isotopes if they appear in at least min_animals
    out of 16 animals and have a score_percentage greater than min_score.
    
    Parameters:
    -----------
    csv_file : str
        Path to the CSV file containing isotope data
    min_animals : int
        Minimum number of animals required (default: 12)
    min_score : float
        Minimum score_percentage required (default: 60)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame containing identified isotopes with summary statistics
    """
    
    # Read the CSV file
    print(f"Reading {csv_file}...")
    df = pd.read_csv(csv_file)
    
    print(f"Total rows in dataset: {len(df)}")
    print(f"Unique samples: {df['sample_id'].nunique()}")
    print(f"Mass difference types: {df['mass_diff_type'].unique()}")
    
    # Filter for rows with score_percentage > min_score
    df_filtered = df[df['score_percentage'] > min_score].copy()
    print(f"\nRows with score_percentage > {min_score}: {len(df_filtered)}")
    
    # Group by m/z pairs to count animals and aggregate statistics
    # Create a unique identifier for each isotope pair
    df_filtered['pair_id'] = df_filtered.apply(
        lambda row: f"{row['mz_1']:.4f}_{row['mz_2']:.4f}_{row['mass_diff_type']}", 
        axis=1
    )
    
    # Count animals per isotope pair
    isotope_stats = []
    
    for pair_id, group in df_filtered.groupby('pair_id'):
        n_animals = group['sample_id'].nunique()
        
        # Only include pairs that appear in at least min_animals
        if n_animals >= min_animals:
            stats = {
                'mz_1': group['mz_1'].iloc[0],
                'mz_2': group['mz_2'].iloc[0],
                'mz_difference': group['mz_difference'].iloc[0],
                'mass_diff_type': group['mass_diff_type'].iloc[0],
                'n_animals': n_animals,
                'mean_score_percentage': group['score_percentage'].mean(),
                'median_score_percentage': group['score_percentage'].median(),
                'min_score_percentage': group['score_percentage'].min(),
                'max_score_percentage': group['score_percentage'].max(),
                'std_score_percentage': group['score_percentage'].std(),
                'animals': ','.join(sorted(group['sample_id'].unique()))
            }
            isotope_stats.append(stats)
    
    # Create results DataFrame
    results_df = pd.DataFrame(isotope_stats)
    
    if len(results_df) > 0:
        # Sort by number of animals (descending) and mean score (descending)
        results_df = results_df.sort_values(
            ['n_animals', 'mean_score_percentage'], 
            ascending=[False, False]
        ).reset_index(drop=True)
    
    return results_df, df_filtered

def save_results(results_df, output_file='identified_isotopes.csv'):
    """Save identified isotopes to CSV file"""
    results_df.to_csv(output_file, index=False)
    print(f"\nResults saved to {output_file}")

def print_summary(results_df, min_animals=12, min_score=60):
    """Print summary statistics"""
    print("\n" + "="*80)
    print(f"ISOTOPE IDENTIFICATION SUMMARY")
    print(f"Criteria: ≥{min_animals} animals AND score_percentage > {min_score}")
    print("="*80)
    
    if len(results_df) == 0:
        print("\nNo isotopes found meeting the criteria.")
        return
    
    print(f"\nTotal isotope pairs identified: {len(results_df)}")
    print(f"\nBreakdown by mass difference type:")
    for mass_type, count in results_df['mass_diff_type'].value_counts().items():
        print(f"  {mass_type}: {count}")
    
    print(f"\nBreakdown by number of animals:")
    for n_animals in sorted(results_df['n_animals'].unique(), reverse=True):
        count = (results_df['n_animals'] == n_animals).sum()
        print(f"  {n_animals} animals: {count} pairs")
    
    print(f"\nScore statistics for identified isotopes:")
    print(f"  Mean score_percentage: {results_df['mean_score_percentage'].mean():.2f}")
    print(f"  Median score_percentage: {results_df['median_score_percentage'].median():.2f}")
    print(f"  Range: {results_df['min_score_percentage'].min():.2f} - {results_df['max_score_percentage'].max():.2f}")
    
    print(f"\nTop 10 isotope pairs by mean score_percentage:")
    print("-"*80)
    top_10 = results_df.head(10)
    for idx, row in top_10.iterrows():
        print(f"{idx+1}. m/z {row['mz_1']:.4f} → {row['mz_2']:.4f} ({row['mass_diff_type']})")
        print(f"   Animals: {row['n_animals']}/16 | Mean score: {row['mean_score_percentage']:.2f}%")
        print()

# Main execution
if __name__ == "__main__":
    # Configuration
    CSV_FILE = "/home/ajarrah/PhD_Thesis/chapter_4/code_combined/2__mz_isotope_matching_results/mz_to_mz_isotope_candidates.csv"
    MIN_ANIMALS = 8
    MIN_SCORE = 40
    OUTPUT_FILE = "/home/ajarrah/PhD_Thesis/chapter_4/code_combined/2__mz_isotope_matching_results/identified_isotopes.csv"

    # Run analysis
    print("Starting isotope identification analysis...\n")
    
    try:
        results_df, filtered_df = identify_isotopes(
            CSV_FILE, 
            min_animals=MIN_ANIMALS, 
            min_score=MIN_SCORE
        )
        
        # Print summary
        print_summary(results_df, MIN_ANIMALS, MIN_SCORE)
        
        # Save results
        if len(results_df) > 0:
            save_results(results_df, OUTPUT_FILE)
            
            # Optional: Save detailed filtered data
            detailed_output = OUTPUT_FILE.replace('.csv', '_detailed.csv')
            filtered_df.to_csv(detailed_output, index=False)
            print(f"Detailed data saved to {detailed_output}")
        
    except FileNotFoundError:
        print(f"Error: File '{CSV_FILE}' not found.")
        print("Please update the CSV_FILE variable with the correct path.")
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

Starting isotope identification analysis...

Reading /home/ajarrah/PhD_Thesis/chapter_4/code_combined/2__mz_isotope_matching_results/mz_to_mz_isotope_candidates.csv...
Total rows in dataset: 12527
Unique samples: 16
Mass difference types: ['Adduct (K)' 'Adduct (NH4)' 'Adduct (Na)' 'Isotope (M+1)' 'Isotope (M+2)']

Rows with score_percentage > 40: 11008

ISOTOPE IDENTIFICATION SUMMARY
Criteria: ≥8 animals AND score_percentage > 40

Total isotope pairs identified: 75

Breakdown by mass difference type:
  Isotope (M+1): 34
  Isotope (M+2): 24
  Adduct (Na): 9
  Adduct (K): 4
  Adduct (NH4): 4

Breakdown by number of animals:
  13 animals: 1 pairs
  11 animals: 15 pairs
  10 animals: 14 pairs
  9 animals: 23 pairs
  8 animals: 22 pairs

Score statistics for identified isotopes:
  Mean score_percentage: 77.42
  Median score_percentage: 79.08
  Range: 40.12 - 98.90

Top 10 isotope pairs by mean score_percentage:
--------------------------------------------------------------------------------

In [16]:
import pandas as pd
import numpy as np

def identify_isotopes(csv_file, min_animals=12, min_score=60):
    """
    Identify m/z values as isotopes if they appear in at least min_animals
    out of 16 animals and have a score_percentage greater than min_score.
    
    Parameters:
    -----------
    csv_file : str
        Path to the CSV file containing isotope data
    min_animals : int
        Minimum number of animals required (default: 12)
    min_score : float
        Minimum score_percentage required (default: 60)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame containing identified isotopes with summary statistics
    """
    
    # Read the CSV file
    print(f"Reading {csv_file}...")
    df = pd.read_csv(csv_file)
    
    print(f"Total rows in dataset: {len(df)}")
    print(f"Unique samples: {df['sample_id'].nunique()}")
    print(f"Mass difference types: {df['mass_diff_type'].unique()}")
    
    # Round m/z values to 3 decimal places
    df['mz_1_rounded'] = df['mz_1'].round(2)
    df['mz_2_rounded'] = df['mz_2'].round(2)
    
    print(f"\nOriginal unique mz_1 values: {df['mz_1'].nunique()}")
    print(f"Rounded unique mz_1 values: {df['mz_1_rounded'].nunique()}")
    print(f"Original unique mz_2 values: {df['mz_2'].nunique()}")
    print(f"Rounded unique mz_2 values: {df['mz_2_rounded'].nunique()}")
    
    # Filter for rows with score_percentage > min_score
    df_filtered = df[df['score_percentage'] > min_score].copy()
    print(f"\nRows with score_percentage > {min_score}: {len(df_filtered)}")
    
    # Group by m/z pairs to count animals and aggregate statistics
    # Create a unique identifier for each isotope pair using rounded values
    df_filtered['pair_id'] = df_filtered.apply(
        lambda row: f"{row['mz_1_rounded']:.3f}_{row['mz_2_rounded']:.3f}_{row['mass_diff_type']}", 
        axis=1
    )
    
    # Count animals per isotope pair
    isotope_stats = []
    
    for pair_id, group in df_filtered.groupby('pair_id'):
        n_animals = group['sample_id'].nunique()
        
        # Only include pairs that appear in at least min_animals
        if n_animals >= min_animals:
            stats = {
                'mz_1': group['mz_1_rounded'].iloc[0],
                'mz_2': group['mz_2_rounded'].iloc[0],
                'mz_difference': group['mz_difference'].mean(),
                'mass_diff_type': group['mass_diff_type'].iloc[0],
                'n_animals': n_animals,
                'mean_score_percentage': group['score_percentage'].mean(),
                'median_score_percentage': group['score_percentage'].median(),
                'min_score_percentage': group['score_percentage'].min(),
                'max_score_percentage': group['score_percentage'].max(),
                'std_score_percentage': group['score_percentage'].std(),
                'animals': ','.join(sorted(group['sample_id'].unique()))
            }
            isotope_stats.append(stats)
    
    # Create results DataFrame
    results_df = pd.DataFrame(isotope_stats)
    
    if len(results_df) > 0:
        # Sort by number of animals (descending) and mean score (descending)
        results_df = results_df.sort_values(
            ['n_animals', 'mean_score_percentage'], 
            ascending=[False, False]
        ).reset_index(drop=True)
    
    return results_df, df_filtered

def save_results(results_df, output_file='identified_isotopes.csv'):
    """Save identified isotopes to CSV file"""
    results_df.to_csv(output_file, index=False)
    print(f"\nResults saved to {output_file}")

def print_summary(results_df, min_animals=12, min_score=60):
    """Print summary statistics"""
    print("\n" + "="*80)
    print(f"ISOTOPE IDENTIFICATION SUMMARY")
    print(f"Criteria: ≥{min_animals} animals AND score_percentage > {min_score}")
    print("="*80)
    
    if len(results_df) == 0:
        print("\nNo isotopes found meeting the criteria.")
        return
    
    print(f"\nTotal isotope pairs identified: {len(results_df)}")
    print(f"\nBreakdown by mass difference type:")
    for mass_type, count in results_df['mass_diff_type'].value_counts().items():
        print(f"  {mass_type}: {count}")
    
    print(f"\nBreakdown by number of animals:")
    for n_animals in sorted(results_df['n_animals'].unique(), reverse=True):
        count = (results_df['n_animals'] == n_animals).sum()
        print(f"  {n_animals} animals: {count} pairs")
    
    print(f"\nScore statistics for identified isotopes:")
    print(f"  Mean score_percentage: {results_df['mean_score_percentage'].mean():.2f}")
    print(f"  Median score_percentage: {results_df['median_score_percentage'].median():.2f}")
    print(f"  Range: {results_df['min_score_percentage'].min():.2f} - {results_df['max_score_percentage'].max():.2f}")
    
    print(f"\nTop 10 isotope pairs by mean score_percentage:")
    print("-"*80)
    top_10 = results_df.head(10)
    for idx, row in top_10.iterrows():
        print(f"{idx+1}. m/z {row['mz_1']:.3f} → {row['mz_2']:.3f} ({row['mass_diff_type']})")
        print(f"   Animals: {row['n_animals']}/16 | Mean score: {row['mean_score_percentage']:.2f}%")
        print()

# Main execution
if __name__ == "__main__":
    # Configuration
    CSV_FILE = "/home/ajarrah/PhD_Thesis/chapter_4/code_combined/2__mz_isotope_matching_results/mz_to_mz_isotope_candidates.csv"
    MIN_ANIMALS = 12
    MIN_SCORE = 60
    OUTPUT_FILE = "/home/ajarrah/PhD_Thesis/chapter_4/code_combined/2__mz_isotope_matching_results/identified_isotopes.csv"

    # Run analysis
    print("Starting isotope identification analysis...\n")
    
    try:
        results_df, filtered_df = identify_isotopes(
            CSV_FILE, 
            min_animals=MIN_ANIMALS, 
            min_score=MIN_SCORE
        )
        
        # Print summary
        print_summary(results_df, MIN_ANIMALS, MIN_SCORE)
        
        # Save results
        if len(results_df) > 0:
            save_results(results_df, OUTPUT_FILE)
            
            # Optional: Save detailed filtered data
            detailed_output = OUTPUT_FILE.replace('.csv', '_detailed.csv')
            filtered_df.to_csv(detailed_output, index=False)
            print(f"Detailed data saved to {detailed_output}")
        
    except FileNotFoundError:
        print(f"Error: File '{CSV_FILE}' not found.")
        print("Please update the CSV_FILE variable with the correct path.")
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

Starting isotope identification analysis...

Reading /home/ajarrah/PhD_Thesis/chapter_4/code_combined/2__mz_isotope_matching_results/mz_to_mz_isotope_candidates.csv...
Total rows in dataset: 12527
Unique samples: 16
Mass difference types: ['Adduct (K)' 'Adduct (NH4)' 'Adduct (Na)' 'Isotope (M+1)' 'Isotope (M+2)']

Original unique mz_1 values: 2017
Rounded unique mz_1 values: 578
Original unique mz_2 values: 2117
Rounded unique mz_2 values: 599

Rows with score_percentage > 60: 9014

ISOTOPE IDENTIFICATION SUMMARY
Criteria: ≥12 animals AND score_percentage > 60

Total isotope pairs identified: 261

Breakdown by mass difference type:
  Isotope (M+1): 154
  Isotope (M+2): 73
  Adduct (Na): 19
  Adduct (K): 11
  Adduct (NH4): 4

Breakdown by number of animals:
  16 animals: 93 pairs
  15 animals: 48 pairs
  14 animals: 52 pairs
  13 animals: 25 pairs
  12 animals: 43 pairs

Score statistics for identified isotopes:
  Mean score_percentage: 77.51
  Median score_percentage: 76.94
  Range: 60

In [3]:
import pandas as pd
import numpy as np

def identify_isotopes(csv_file, min_animals=12, min_mean_score=60):
    """
    Identify m/z values as isotopes if they appear in at least min_animals
    out of 16 animals and have a mean_score_percentage greater than min_mean_score.
    
    Parameters:
    -----------
    csv_file : str
        Path to the CSV file containing isotope data
    min_animals : int
        Minimum number of animals required (default: 12)
    min_mean_score : float
        Minimum mean_score_percentage required (default: 60)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame containing identified isotopes with summary statistics
    """
    
    # Read the CSV file
    print(f"Reading {csv_file}...")
    df = pd.read_csv(csv_file)
    
    print(f"Total rows in dataset: {len(df)}")
    print(f"Unique samples: {df['sample_id'].nunique()}")
    print(f"Mass difference types: {df['mass_diff_type'].unique()}")
    
    # Round m/z values to 3 decimal places
    df['mz_1_rounded'] = df['mz_1'].round(2)
    df['mz_2_rounded'] = df['mz_2'].round(2)
    
    print(f"\nOriginal unique mz_1 values: {df['mz_1'].nunique()}")
    print(f"Rounded unique mz_1 values: {df['mz_1_rounded'].nunique()}")
    print(f"Original unique mz_2 values: {df['mz_2'].nunique()}")
    print(f"Rounded unique mz_2 values: {df['mz_2_rounded'].nunique()}")
    
    # Create a unique identifier for each isotope pair using rounded values
    df['pair_id'] = df.apply(
        lambda row: f"{row['mz_1_rounded']:.3f}_{row['mz_2_rounded']:.3f}_{row['mass_diff_type']}", 
        axis=1
    )
    
    # Count animals per isotope pair and calculate mean score
    isotope_stats = []
    
    for pair_id, group in df.groupby('pair_id'):
        n_animals = group['sample_id'].nunique()
        mean_score = group['score_percentage'].mean()
        
        # Only include pairs that appear in at least min_animals AND have mean_score >= min_mean_score
        if n_animals >= min_animals and mean_score >= min_mean_score:
            stats = {
                'mz_1': group['mz_1_rounded'].iloc[0],
                'mz_2': group['mz_2_rounded'].iloc[0],
                'mz_difference': group['mz_difference'].mean(),  # Average the difference since rounded values may vary slightly
                'mass_diff_type': group['mass_diff_type'].iloc[0],
                'n_animals': n_animals,
                'mean_score_percentage': mean_score,
                'median_score_percentage': group['score_percentage'].median(),
                'min_score_percentage': group['score_percentage'].min(),
                'max_score_percentage': group['score_percentage'].max(),
                'std_score_percentage': group['score_percentage'].std(),
                'animals': ','.join(sorted(group['sample_id'].unique()))
            }
            isotope_stats.append(stats)
    
    # Create results DataFrame
    results_df = pd.DataFrame(isotope_stats)
    
    if len(results_df) > 0:
        # Sort by number of animals (descending) and mean score (descending)
        results_df = results_df.sort_values(
            ['n_animals', 'mean_score_percentage'], 
            ascending=[False, False]
        ).reset_index(drop=True)
    
    print(f"\nIsotope pairs with ≥{min_animals} animals AND mean_score_percentage ≥{min_mean_score}: {len(results_df)}")
    
    return results_df

def save_results(results_df, output_file='identified_isotopes.csv'):
    """Save identified isotopes to CSV file"""
    results_df.to_csv(output_file, index=False)
    print(f"\nResults saved to {output_file}")

def print_summary(results_df, min_animals=12, min_mean_score=60):
    """Print summary statistics"""
    print("\n" + "="*80)
    print(f"ISOTOPE IDENTIFICATION SUMMARY")
    print(f"Criteria: ≥{min_animals} animals AND mean_score_percentage ≥{min_mean_score}")
    print("="*80)
    
    if len(results_df) == 0:
        print("\nNo isotopes found meeting the criteria.")
        return
    
    print(f"\nTotal isotope pairs identified: {len(results_df)}")
    print(f"\nBreakdown by mass difference type:")
    for mass_type, count in results_df['mass_diff_type'].value_counts().items():
        print(f"  {mass_type}: {count}")
    
    print(f"\nBreakdown by number of animals:")
    for n_animals in sorted(results_df['n_animals'].unique(), reverse=True):
        count = (results_df['n_animals'] == n_animals).sum()
        print(f"  {n_animals} animals: {count} pairs")
    
    print(f"\nScore statistics for identified isotopes:")
    print(f"  Mean score_percentage: {results_df['mean_score_percentage'].mean():.2f}")
    print(f"  Median score_percentage: {results_df['median_score_percentage'].median():.2f}")
    print(f"  Range: {results_df['mean_score_percentage'].min():.2f} - {results_df['mean_score_percentage'].max():.2f}")
    
    print(f"\nTop 10 isotope pairs by mean score_percentage:")
    print("-"*80)
    top_10 = results_df.head(10)
    for idx, row in top_10.iterrows():
        print(f"{idx+1}. m/z {row['mz_1']:.3f} → {row['mz_2']:.3f} ({row['mass_diff_type']})")
        print(f"   Animals: {row['n_animals']}/16 | Mean score: {row['mean_score_percentage']:.2f}%")
        print()

# Main execution
if __name__ == "__main__":
    # Configuration
    CSV_FILE = "/home/ajarrah/PhD_Thesis/chapter_4/code_combined/2__mz_isotope_matching_results/mz_to_mz_isotope_candidates.csv"
    MIN_ANIMALS = 1
    MIN_MEAN_SCORE = 1
    OUTPUT_FILE = "/home/ajarrah/PhD_Thesis/chapter_4/code_combined/2__mz_isotope_matching_results/identified_isotopes.csv"

    # Run analysis
    print("Starting isotope identification analysis...\n")
    
    try:
        results_df = identify_isotopes(
            CSV_FILE, 
            min_animals=MIN_ANIMALS, 
            min_mean_score=MIN_MEAN_SCORE
        )
        
        # Print summary
        print_summary(results_df, MIN_ANIMALS, MIN_MEAN_SCORE)
        
        # Save results
        if len(results_df) > 0:
            save_results(results_df, OUTPUT_FILE)
        
    except FileNotFoundError:
        print(f"Error: File '{CSV_FILE}' not found.")
        print("Please update the CSV_FILE variable with the correct path.")
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

Starting isotope identification analysis...

Reading /home/ajarrah/PhD_Thesis/chapter_4/code_combined/2__mz_isotope_matching_results/mz_to_mz_isotope_candidates.csv...
Total rows in dataset: 12527
Unique samples: 16
Mass difference types: ['Adduct (K)' 'Adduct (NH4)' 'Adduct (Na)' 'Isotope (M+1)' 'Isotope (M+2)']

Original unique mz_1 values: 2017
Rounded unique mz_1 values: 578
Original unique mz_2 values: 2117
Rounded unique mz_2 values: 599

Isotope pairs with ≥1 animals AND mean_score_percentage ≥1: 1492

ISOTOPE IDENTIFICATION SUMMARY
Criteria: ≥1 animals AND mean_score_percentage ≥1

Total isotope pairs identified: 1492

Breakdown by mass difference type:
  Isotope (M+1): 583
  Isotope (M+2): 442
  Adduct (Na): 248
  Adduct (NH4): 144
  Adduct (K): 75

Breakdown by number of animals:
  16 animals: 234 pairs
  15 animals: 56 pairs
  14 animals: 62 pairs
  13 animals: 36 pairs
  12 animals: 51 pairs
  11 animals: 122 pairs
  10 animals: 65 pairs
  9 animals: 71 pairs
  8 animals: 5